In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# after47 同状态梯度重复性与精度对照
直接复用 after47-diagnostic01 保存的 latent、完整 scheduler 历史、目标及固定正负增量，不重建前缀。现有混合精度求梯度两次，FP32 算术求梯度一次；两种精度各计算一次无梯度基线与正负候选。不会选择候选，不改写 run08。


In [ ]:
from pathlib import Path
import sys, subprocess, json
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_BRANCH = 'dev/生成端公共关系载体/二维图像统计-持续生成约束'
SOURCE = Path('/content/public_statistic_phase2_source')
if SOURCE.exists(): raise FileExistsError('Use a fresh runtime; preserve existing source')
# 获取已发布开发分支，并记录实际使用的提交。
subprocess.run(['git','init',str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'remote','add','origin',REPOSITORY_URL],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','--depth','1','origin',SOURCE_BRANCH],check=True)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','FETCH_HEAD'],check=True)
print('Source:',subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip())


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','diffusers','transformers','accelerate','ftfy','sentencepiece','safetensors','huggingface_hub','numpy','Pillow'],check=True)
subprocess.run(['ffmpeg','-version'],check=True)
# Keep Colab CUDA PyTorch. Actual versions are recorded by the worker.


## 已确认范围
总计 36 次 Transformer、9 次 VAE、3 次 backward；另计最多 360 block / 39 chunk 重算。0 MP4，1800 秒墙钟，零重试。无显卡型号白名单或人为显存配额，计算路径需要 CUDA/BF16。
FP32 对照将同一已舍入 Transformer 权重与提示嵌入提升至 FP32，不重载不同权重；关闭 autocast 和 TF32。VAE、solver 和保存的历史值不变。这是算术精度对照，不是原始全 FP32 模型实验；真实 GPU 资源与结果待本次运行验证。


In [ ]:
BASE=Path('/content/drive/MyDrive/Video-WM/public-statistic-phase2')
CONFIG=BASE/'run08/config.json'
REFERENCE=BASE/'after47-diagnostic01/after47_state_and_gradient.pt'
OUTPUT=BASE/'after47-precision01'
for path in (CONFIG,REFERENCE):
    if not path.is_file(): raise FileNotFoundError(path)
if OUTPUT.exists(): raise FileExistsError(str(OUTPUT))
print('Source config:',CONFIG.read_text())
print('Budget: 36 transformer / 9 VAE / 3 backward; 360 block / 39 chunk replay; 0 MP4; 1800 seconds')


In [ ]:
import os, signal
command=[sys.executable,'-m','experiments.public_statistic.run_after47_precision','--config',str(CONFIG),'--reference',str(REFERENCE),'--output',str(OUTPUT)]
process=subprocess.Popen(command,cwd=SOURCE,start_new_session=True)
try:
    returncode=process.wait()
except BaseException:
    # Cancel the launcher; it forwards cancellation and preserves worker results.
    try: process.send_signal(signal.SIGTERM)
    except ProcessLookupError: pass
    try: process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        try: os.killpg(process.pid,signal.SIGKILL)
        except ProcessLookupError: pass
        process.wait()
    raise
print('launcher exit',returncode)
print((OUTPUT/'result.json').read_text() if (OUTPUT/'result.json').exists() else 'No result file')
if returncode: raise subprocess.CalledProcessError(returncode,command)


## 结果
输出到 after47-precision01。回传 result.json、execution_exit.json、loaded_model.json、runtime.json、recomputation.json、execution.log。三个 gradient .pt 文件保留完整梯度，固定输入仍引用原始快照。报告逐元素差异、相对 L2、余弦、同一 Δ 的 g·Δ、基线与正负 loss；不依据最佳方向或精度自动裁决方法成功。
